In [14]:

import numpy as np
import matplotlib.pyplot as plt
file_path = 'SNR_time.txt'
data = np.loadtxt(file_path, skiprows=1)



SNR2 =15 #5 sigma deteciton
#G=0.32 *13,  X1= 93(for legacy GMRT) #TT_rec+T_sky: kelvin --> Band3 (Best RMS sensitivity 0.04 mJy with all antennas) (T = 93 ), band4 (T =60+10=70) (Best RMS sensitivity 0.02 mJy)
#for uGMRT
nu1 = 400  #MHz, central freq of band3
nu2 = 650  #MHz, Central frequency of band4
#for band 3
a0 = -3.94269512191062 
a1 = 609.220731323399 * 1e-4
a2 = -388.278427948319 * 1e-6
a3 = 130.788311514484 * 1e-8
a4 = -245.688427095004 *1e-11
a5 = 24.4226778956594 * 1e-13
a6 = -100.460621956708 * 1e-17
X1 = a0 + a1 * nu1 + a2 * nu1**2 + a3 * nu1**3 + a4 * nu1**4 +a5 *nu1**5 + a6 * nu1**6#Jy-1, nu in MHz = G/T_sys

B=200e6 #Hz

n=20 #no of antenna
P=0.002600999083 #sec
W= 0.381e-3  #sec (BasicallyTaking 10% of the time period, here I have calculate it)

#Best rms sensitivity achieved in band-3 is 0.0.mJy

SNR =data[:,0]
t=data[:,1] #sec

#Calculation of S_pulsar (SNR = 5 for detection)
def S_mean(SNR2,X1,W,n,t,B,P):
    S_mean = SNR2 * 1e3 *  (W**0.5) / (X1 * ((2*t*n**2*B)**0.5) * ((P-W)**0.5) )  #converted into Jy to SI units and in units of mJy
    return S_mean
s = S_mean(SNR2,X1,W,n,t,B,P)
print("The S_mean value (for sigma = 15,) is" , S_mean(SNR2,X1,W,n,t,B,P),"mJy")

def S_pulsar(SNR,X1,W,n,t,B,P): 
    S_pulsar = SNR * 1e3 * (W**0.5) / ( X1 * ((t*n**2 *B)**0.5) * ((P-W)**0.5) )  #converted into Jy to SI units and in units of mJy
    return S_pulsar
S = S_pulsar(SNR2,X1,W,n,t,B,P)
total_S_pulsar = S_pulsar(SNR,X1,W,n,t,B,P)
print("The S_pulsar value is" , total_S_pulsar ,"mJy")
print(f"Maximum Value of pulsar flux density is:: {np.max(total_S_pulsar)}")
print(f"Average Value of pulsar flux density is:: {np.average(total_S_pulsar)}")

#for single pulse search, the flux density formula is: see etc_help.pdf in the same folder for the equations  #http://www.ncra.tifr.res.in:8081/~secr-ops/etc/etc_help.pdf
def S_SP(SNR_SP,X1,W,n,B,P):   #here t=P is the period of the pulsar
    S_SP = SNR_SP * 1e3 / ( X1 * ((2*W*n**2 *B)**0.5) )# * (W**0.5) )  #converted into Jy to SI units and in units of mJy
    return S_SP
S = S_SP(SNR2,X1,W,n,B,P)
S_single_pulse = S_SP(15,X1,W,n,B,P)
print(f"The flux density of the single pulse is:: {S_single_pulse}")




def TOA(X1,P,B,t,S_mean):   #from calculated S_mean
    TOA = 10**6 * P * (W/P)**1.5 / ( (t*B)**0.5 * S_mean * X1)  #divided with 13**0.5 for array of 13 antenna.# in microseconds units
    return TOA
#rms_toa =  TOA(S_sys,P,B,t,S_mean)
print("The RMS_TOA value is",TOA(X1,P,B,t,s), "in micro seconds")
print(" All these values are in relative flux unit, for absolute flux values, calibrator needde or determine by P_off and P_on regoin method.")

#Calculation for time of OBS for 4 non-detection epochs
def t_obs(X1, B, P, W):
    return (5/(X1*75e-4))**2 * W/ (14**2 * B * (P-W)) 
print('The minimum time of observation for 5 sigma dtection of 7.5mJy pulsar in band-3 is',t_obs(X1, B, P, W), 'seconds' )

def RMS_TOA(X1, B, P, W): #from S_mean = 7.5 mJy
    return W**1.5 / ((2400*B)**0.5 * 75e-4 * X1 * P**0.5) #mean flux density in band-3 is 7.t_obs(X1, B, P, W)5 mjy from Bhattacharyya et al 2021, avg time _obs=2400sec
print("The value of RMS_TOA is given by",RMS_TOA(X1, B, P, W)* 1e6,"im micro-sec units")

The S_mean value (for sigma = 15,) is [8.86805080e-02 7.82088568e-02 8.21021968e-02 7.41954362e-02
 8.43711225e-02 9.57858963e-02 6.42551326e-02 6.42551326e-02
 6.77308568e-02 6.77308568e-02 8.29530195e-02 7.96833750e+01
 7.96833750e+01] mJy
The S_pulsar value is [  0.45483179   0.55302013   0.45515136   0.44489553   0.4454568
   0.48675909   0.60337999   0.59671615   0.65006695   0.64495837
   0.46768896 112.68930957  90.15144766] mJy
Maximum Value of pulsar flux density is:: 112.68930957334413
Average Value of pulsar flux density is:: 16.04951402722201
The flux density of the single pulse is:: 502.5615304571564
The RMS_TOA value is [0.66372016 0.66372016 0.66372016 0.66372016 0.66372016 0.66372016
 0.66372016 0.66372016 0.66372016 0.66372016 0.66372016 0.66372016
 0.66372016] in micro seconds
 All these values are in relative flux unit, for absolute flux values, calibrator needde or determine by P_off and P_on regoin method.
The minimum time of observation for 5 sigma dtection of 7.5

In [2]:
#calculation of magnetic field
P =  0.00260083478563372  #in second
del_P=  9.1832e-17 
P_dot= 1.6827579188045e-20    
del_P_dot= 6.2406e-25   
E_1 = 3.8e34
tau_1 = 2.45e9
B_1 = 2.12e8 



#Energy loss rate calculation:
E_dot = 3.95 * 1e31 * (P_dot/ 1e-15) / P**3 # in units of erg/s #P_dot in units of 1e15
del_E_dot =E_dot * np.sqrt((del_P_dot/(P_dot))**2 +(3* del_P/P)**2)
print("E_dot=",E_dot, "erg s-1")
print("error E_dot=",del_E_dot, "erg s-1")

#Charecteristics age
tau = 15.8 * P / (P_dot/ 1e-15) #P_dot in units of 1e15
del_tau = tau * np.sqrt((del_P/P)**2+(del_P_dot/(P_dot))**2)#multiplying 1e6 for Myr units of ttau
print("tau =",tau, "Myr")
print("error tau =",del_tau, "Myr")


B = 10**12 * (P * (P_dot/ 1e-15))**0.5 #in Guass units dividing 1e-15 in P
del_B = B * np.sqrt((0.5*del_P/P)**2+(0.5*del_P_dot/(P_dot))**2)
print("B =",B, "Gauss")
print("errorB =",del_B, "Gauss")

# from P & P_dot, transverse velocity calculation
V_T = np.sqrt(P_dot * 3 * 1e5 * 2.5 *3.086e+16 / P)
print("xx", V_T)

#stability ncalculation
nu_dot =  -2.49064176742383e-15
nu =  384.491935253077 
nu_double_dot = 3.4 * nu_dot**2 / nu
print("Value of spin double derivative",nu_double_dot , "Hz")
stability = np.log10(nu_double_dot* (1e8 )**3 / (6* nu))
print("satbility factor", stability)

print("relative change values E_dot",( E_1 - E_dot)/E_1)
print("relative change values tau", "tau",(tau_1 - tau*1e6)/tau_1)
print("relative change values B",  "magnetic field", (B_1 - B)/B_1)


E_dot= 3.7781608476881235e+34 erg s-1
error E_dot= 1.4011516643364407e+30 erg s-1
tau = 2442.014335740406 Myr
error tau = 0.09056344048850733 Myr
B = 209202660.85849258 Gauss
errorB = 3879.19768721405 Gauss
xx 386.97510920737136
Value of spin double derivative 5.485474693371725e-32 Hz
satbility factor -10.623824269428598
relative change values E_dot 0.0057471453452306994
relative change values tau tau 0.003259454799834271
relative change values B magnetic field 0.013194995950506683


In [3]:
T_sys = 93
def TOA(T_sys,B):
    TOA = 1e6 * T_sys /(t*B*n)**0.5   #divided with 13**0.5 for array of 13 antenna.# in microseconds units
    return TOA
rms_toa = TOA(T_sys,B)
print("The RMS_TOA value is",TOA(T_sys,B), "in micro seconds")


The RMS_TOA value is [3.13743474e+01 2.76695736e+01 2.90470015e+01 2.62496623e+01
 2.98497265e+01 3.38881684e+01 2.27328744e+01 2.27328744e+01
 2.39625537e+01 2.39625537e+01 2.93480147e+01 2.81912446e+04
 2.81912446e+04] in micro seconds


In [4]:
#calculation of Transverse velocity
d= 1.5  #in kpc units; 1.8(NE2001), 2.5 (YMW)
proper_motion = 7.1348   #in units of mas/yr
del_proper_motion = 0.46519  #mas/yr

V_trans = 4.74 * proper_motion *d

del_V_trans = V_trans * np.sqrt((del_proper_motion/proper_motion)**2 + (0.1)**2)
print(V_trans)
print(del_V_trans)

50.72842800000001
6.055848105510298


In [5]:
pip install "numpy<2.0"


Note: you may need to restart the kernel to use updated packages.


In [6]:
import numpy as np
import matplotlib.pyplot as plt

# Input paamters
#DM_value = 48.26341         # Dispersion Measure in pc/cm^3 J0248+4230
DM_value = 50.67
frequency_value = 0.4    # Observing frequency in GHz

def calculate_scattering_timescale(DM, freq):
    """
    Calculate scattering timescale (pulse broadening time) using Bhat et al. (2004) empirical relation.
    
    Parameters:
        DM (float): Dispersion Measure in pc/cm^3
        freq (float): Observing frequency in GHz
        
    Returns:
        float: Scattering timescale in milliseconds
    """
    log_tau = -6.46 + 0.154 * np.log10(DM) + 1.07 * (np.log10(DM))**2 - 3.86 * np.log10(freq)
    return 10**log_tau  # Result in milliseconds

# Calculate scattering time
scattering_timescale = calculate_scattering_timescale(DM_value, frequency_value)

# Print result with better formatting
print(f"Scattering Time Scale for DM = {DM_value} pc/cm³ at {frequency_value} GHz: {scattering_timescale:.4e} ms")


Scattering Time Scale for DM = 50.67 pc/cm³ at 0.4 GHz: 2.8066e-02 ms


In [7]:
def compute_value(P, x):
    return (360 * 0.96e-6 / P1 )

# Example usage:
P = 1.877
P1 = P * 0.001
# example value in milliseconds
x = 0.37  # example number of bins
result = compute_value(P, x)

print("Computed value (in ms):", result)


Computed value (in ms): 0.18412360149174212


In [8]:
#Intra channel dispersion value
v_1 = 650 #MHz units
v_2 = 650 + 200/4096 # BW= 200 MHz, freq channel = 4096
DM = 15   #pc/cm^3
def del_t(v_1,v_2,DM):
    return 4.15 * 1e6 * (v_1**-2 - v_2**-2) * DM #in ms units
print(del_t(v_1,v_2,DM))


0.022133515288779892
